# SKKU1398 운동기구 데이터셋 v1 구축

원본 보존·자동 검사·중복 그룹화·시각 검수 승인 후 내보내기. 모델 학습은 실행하지 않습니다.
기존 원본은 Colab 임시 저장소에 있을 수 있습니다. 복구 시 raw_sources 압축을 먼저 풀어야 합니다.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")
from pathlib import Path
ROOT=Path("/content/drive/MyDrive/SKKU_1398_Fitness_ML")


In [ ]:
%pip -q install pillow pyyaml ImageHash


## 원본 복구가 필요한 경우
저장된 출처별 ZIP을 기존 경로로 복구합니다. 검수 결과와 승인 파일은 덮어쓰지 않습니다.


In [ ]:
import json, zipfile
sources=json.loads((ROOT/"02_dataset/SKKU1398_equipment_v1/manifests/sources.json").read_text())
for s in sources:
    dest=Path(s["local_root"])
    if not (dest/"data.yaml").exists():
        with zipfile.ZipFile(ROOT/"02_dataset/SKKU1398_equipment_v1"/s["archive"]) as z:
            for member in z.namelist():
                assert not Path(member).is_absolute() and ".." not in Path(member).parts
            z.extractall(dest)
    print(s["source_id"],(dest/"data.yaml").exists())


## 자동 구축
이미 완료한 출처는 검사 체크포인트를 재사용합니다. 별도 프로세스 잠금으로 중복 실행을 방지합니다.


In [ ]:
import subprocess, sys
subprocess.run([sys.executable,str(ROOT/"01_notebooks/build_equipment_v1.py")],check=True)
subprocess.run([sys.executable,str(ROOT/"01_notebooks/package_review_material.py")],check=True)


## 시각 검수
review/review_pack_v1.zip의 사진과 박스를 실제로 검수한 뒤 visual_decisions.json에 승인·보류·제외, 이유, 검수 근거를 기록합니다.
자동 검사 통과 또는 데이터셋 클래스명만으로 승인하지 않습니다.
승인 내보내기는 별도 export_reviewed_v1.py로 수행합니다. 이전 배포 파일이 있으면 새 버전 경로를 사용해야 하며 이 노트북의 모두 실행으로 자동 재배포하지 않습니다.
